# Build 6 — Controlled XGBoost Hyperparameter Tuning

Kaggle Playground Series S6E8 — Predicting Smartphone Addiction

**Objective:** how much additional ROC AUC can be obtained from the
current best XGBoost model (E006) through controlled hyperparameter
tuning, keeping the accepted feature set and validation framework fixed?

**Frozen for this build:** feature set (raw predictors + `screen_residual`),
`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`, preprocessing,
categorical handling, `id` exclusion.

**Only variable:** XGBoost model configuration.

**Staged tuning order:** iteration budget / learning rate -> tree
complexity -> sampling -> regularization -> limited joint refinement
(only if warranted). Screening (single frozen fold, via
`src.tuning.screen_xgboost_single_fold`) is used to eliminate poor
regions cheaply; every formal claim comes only from full 5-fold CV via
`src.tuning.run_xgboost_trial`.

**Explicitly out of scope:** ensembling, blending, rank averaging,
stacking, final submission strategy, new/generator-derived features,
broad brute-force search, LightGBM tuning, CatBoost retuning (one
optional iteration-budget diagnostic only, not run in this notebook —
see `CONTEXT.md`).

## 1. Setup and frozen controls

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from src.config import (
    DELIVERABLES_DIR,
    EXPERIMENTS_DIR,
    ID_COLUMN,
    OUTPUTS_DIR,
    SAMPLE_SUBMISSION_PATH,
    TARGET_COLUMN,
    TEST_PATH,
    TRAIN_PATH,
)
from src.features import add_screen_residual
from src.preprocessing import build_boosting_frame
from src.submission_validation import validate_submission
from src.tuning import (
    E006_XGB_PARAMS,
    paired_fold_deltas,
    run_xgboost_trial,
    screen_xgboost_single_fold,
    tuning_result_row,
)

pd.set_option("display.max_columns", 50)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

X = add_screen_residual(build_boosting_frame(train)).drop(columns=["component_sum"])
y = train[TARGET_COLUMN]
X_test = add_screen_residual(build_boosting_frame(test)).drop(columns=["component_sum"])

print("train shape:", train.shape, "X shape:", X.shape)
print("test shape:", test.shape, "X_test shape:", X_test.shape)

train shape: (691369, 14) X shape: (691369, 13)
test shape: (296302, 13) X_test shape: (296302, 13)


In [2]:
experiments_path = EXPERIMENTS_DIR / "experiments.csv"
tuning_results_path = OUTPUTS_DIR / "xgboost_tuning_results.csv"

experiments = pd.read_csv(experiments_path)
e006_row = experiments.loc[experiments["experiment_id"] == "E006"].iloc[0]
E006_CV_MEAN, E006_CV_STD = float(e006_row["cv_mean"]), float(e006_row["cv_std"])
E006_FOLDS = [float(e006_row[f"fold_{i}_auc"]) for i in range(1, 6)]

print(f"E006 (XGBoost, frozen control): CV mean {E006_CV_MEAN:.5f}, std {E006_CV_STD:.5f}")
print(f"  fold scores: {E006_FOLDS}")
print("E006 params:", E006_XGB_PARAMS)

E006 (XGBoost, frozen control): CV mean 0.96445, std 0.00056
  fold scores: [0.96374, 0.96438, 0.96453, 0.96543, 0.96416]
E006 params: {'objective': 'binary:logistic', 'eval_metric': 'auc', 'learning_rate': 0.1, 'max_depth': 6, 'min_child_weight': 1, 'subsample': 0.9, 'colsample_bytree': 0.9, 'gamma': 0.0, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'n_estimators': 800, 'tree_method': 'hist', 'random_state': 42}


## 2. Phase 0 — E006 reconstruction and iteration-cap diagnosis

**Why:** Build 3/4 XGBoost runs (E004, E006) frequently hit or nearly hit
the 800-iteration training cap (E006 `best_iterations`: `[799, 794, 799,
794, 794]` — 4/5 folds at or one below the cap). This is a resource-
practicality cap, not evidence of convergence, and motivates testing
whether E006 was materially constrained by it.

`E006_XGB_PARAMS` (`src/tuning.py`) is reconstructed to exactly match the
config recorded in `experiments/experiments.csv`'s E006 row. Rerunning it
through the full 5-fold CV harness here both confirms the reconstruction
and re-derives its out-of-fold predictions (not previously saved to
disk) for the Section 9 correlation diagnostic.

In [3]:
t0 = time.time()
e006_reconstructed = run_xgboost_trial(E006_XGB_PARAMS, X, y)
e006_recon_elapsed = time.time() - t0

print(f"\nCV mean: {e006_reconstructed.cv_mean:.5f} (recorded: {E006_CV_MEAN:.5f})")
print(f"CV std:  {e006_reconstructed.cv_std:.5f} (recorded: {E006_CV_STD:.5f})")
print(f"best_iterations: {e006_reconstructed.best_iterations}")
print(f"elapsed: {e006_recon_elapsed:.1f}s")

assert abs(e006_reconstructed.cv_mean - E006_CV_MEAN) < 1e-4, "E006 reconstruction does not match recorded CV mean"
print("\nReconstruction confirmed: matches experiments.csv E006 row.")

fold 1: ROC AUC = 0.96374


fold 2: ROC AUC = 0.96438


fold 3: ROC AUC = 0.96453


fold 4: ROC AUC = 0.96543


fold 5: ROC AUC = 0.96416

CV mean: 0.96445 (recorded: 0.96445)
CV std:  0.00056 (recorded: 0.00056)
best_iterations: [799, 799, 799, 798, 799]
elapsed: 862.2s

Reconstruction confirmed: matches experiments.csv E006 row.


## 3. Phase 1 — Iteration budget / learning-rate screening

Single-fold screens (fold 1 of the frozen splitter) across
`learning_rate` in {0.03, 0.05, 0.07} with a proportionally raised
`n_estimators` ceiling, plus an E006-config baseline screen to confirm
the screening harness reproduces the full-CV fold-1 score exactly.

**Note on execution:** these 4 single-fold screens (and the Phase
2/3/4 screens later in this notebook) were run interactively earlier
in this same investigation, using this exact code path
(`src.tuning.screen_xgboost_single_fold`, same seed, same deterministic
splitter) — see `outputs/xgboost_tuning_results.csv` for the recorded
runs. To keep this notebook's total runtime practical (~15 single-fold
and full-CV XGBoost fits would otherwise take 2+ hours on CPU), the 10
screening trials are read from that tracker below rather than
recomputed live; Phase 0 (E006 reconstruction, Section 2) and Phase 1's
promoted candidate (E010's full CV, Section 4) *are* run live in this
notebook as reproducibility anchors — both exactly match their recorded
values (see the assertion in Section 2 and the comparison in Section
4).

In [4]:
def append_tuning_row(row: dict) -> None:
    existing = pd.read_csv(tuning_results_path)
    if row["trial_or_experiment_id"] in existing["trial_or_experiment_id"].astype(str).values:
        print(f"{row['trial_or_experiment_id']} already recorded — skipping append (idempotent re-run).")
        return
    import csv
    with open(tuning_results_path, "r", newline="", encoding="utf-8") as f:
        fieldnames = next(csv.reader(f))
    with open(tuning_results_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow(row)
    print(f"Row appended for {row['trial_or_experiment_id']}")


def show_stage(stage: str) -> pd.DataFrame:
    """Displays already-recorded screening rows for `stage` from the tuning tracker."""
    df = pd.read_csv(tuning_results_path)
    subset = df.loc[df["stage"] == stage].reset_index(drop=True)
    assert len(subset) > 0, f"no recorded rows for stage={stage!r} — was this stage run and recorded?"
    return subset


phase1_screens = show_stage("phase1_screening")
phase1_screens

,trial_or_experiment_id,stage,formal_experiment,learning_rate,n_estimators,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,gamma,cv_mean,cv_std,delta_vs_control,folds_improved,mean_best_iteration,runtime_minutes,decision
0,screen-p1-baseline,phase1_screening,NaN,0.10,800,6,1,0.9,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,799.0,2.1,single-fold screen only (fold 1); reproduces E...
1,screen-p1-lr0.05,phase1_screening,NaN,0.05,2000,6,1,0.9,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,1994.0,7.2,single-fold screen; fold_auc=0.96417 (+0.00043...
2,screen-p1-lr0.07,phase1_screening,NaN,0.07,1400,6,1,0.9,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,1385.0,4.0,single-fold screen; fold_auc=0.96414 (+0.00040...
3,screen-p1-lr0.03,phase1_screening,NaN,0.03,3200,6,1,0.9,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,3199.0,9.1,single-fold screen; fold_auc=0.96420 (+0.00046...


## 4. Phase 1 — E010 full 5-fold CV (formal experiment)

`learning_rate=0.05` (screened above) is promoted to a full 5-fold CV
with a proportionally raised ceiling (`n_estimators=2500`), run once
with `X_test` supplied so the same trial also produces the fold-averaged
test predictions used later for the final submission (Section 11).

In [5]:
E010_PARAMS = {**E006_XGB_PARAMS, "learning_rate": 0.05, "n_estimators": 2500}

t0 = time.time()
e010_result = run_xgboost_trial(E010_PARAMS, X, y, X_test=X_test)
e010_elapsed = time.time() - t0

print(f"\nCV mean: {e010_result.cv_mean:.5f}")
print(f"CV std:  {e010_result.cv_std:.5f}")
print(f"best_iterations: {e010_result.best_iterations}")
print(f"elapsed: {e010_elapsed:.1f}s")

e010_deltas = paired_fold_deltas(e010_result.fold_scores, E006_FOLDS)
print("\npaired fold deltas vs E006:", e010_deltas)

fold 1: ROC AUC = 0.96429


fold 2: ROC AUC = 0.96500


fold 3: ROC AUC = 0.96509


fold 4: ROC AUC = 0.96585


fold 5: ROC AUC = 0.96475

CV mean: 0.96499
CV std:  0.00051
best_iterations: [2485, 2300, 2445, 2429, 2294]
elapsed: 2613.2s

paired fold deltas vs E006: {'mean_delta': 0.0005465162018022385, 'min_delta': 0.000417258818789068, 'max_delta': 0.000623182449247639, 'folds_improved': 5, 'fold_deltas': [0.0005507979655103856, 0.000623182449247639, 0.0005551321015351274, 0.000417258818789068, 0.0005862096739289724]}


In [6]:
def append_experiment_row(row: dict) -> None:
    existing = pd.read_csv(experiments_path)
    if row["experiment_id"] in existing["experiment_id"].astype(str).values:
        print(f"{row['experiment_id']} already recorded — skipping append (idempotent re-run).")
        return
    import csv
    with open(experiments_path, "r", newline="", encoding="utf-8") as f:
        fieldnames = next(csv.reader(f))
    with open(experiments_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow(row)
    print(f"Row appended for {row['experiment_id']}")


CV_METHOD = "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)"
SHARED_PREPROCESSING = (
    "raw numeric predictors (missing values preserved, native handling); "
    "categorical predictors: explicit Missing category, category dtype "
    "(native handling); id excluded; XGBoost enable_categorical=True"
)
E010_PARAMS_STR = (
    "XGBClassifier(objective=binary:logistic, eval_metric=auc, n_estimators=2500, "
    "learning_rate=0.05, max_depth=6, min_child_weight=1, subsample=0.9, "
    "colsample_bytree=0.9, reg_alpha=0.0, reg_lambda=1.0, gamma=0.0, "
    "early_stopping_rounds=50, tree_method=hist, random_state=42)"
)

e010_row = {
    "experiment_id": "E010",
    "date": "2026-08-21",
    "model": "XGBClassifier",
    "feature_set": "raw + screen_residual",
    "hypothesis": (
        "E006 (and E004 before it) frequently hit or nearly hit the 800-iteration "
        "training cap (best_iterations 794-799 in 4/5 E006 folds), suggesting the model "
        "was iteration-constrained rather than converged. Lowering learning_rate and "
        "raising the iteration ceiling (with the same early_stopping_rounds=50) should "
        "let early stopping find a true best_iteration instead of being cut off by the "
        "cap, and should therefore improve CV."
    ),
    "preprocessing": SHARED_PREPROCESSING,
    "cv_method": CV_METHOD,
    "seed": 42,
    "parameters": E010_PARAMS_STR,
    "fold_1_auc": round(e010_result.fold_scores[0], 5),
    "fold_2_auc": round(e010_result.fold_scores[1], 5),
    "fold_3_auc": round(e010_result.fold_scores[2], 5),
    "fold_4_auc": round(e010_result.fold_scores[3], 5),
    "fold_5_auc": round(e010_result.fold_scores[4], 5),
    "cv_mean": round(e010_result.cv_mean, 5),
    "cv_std": round(e010_result.cv_std, 5),
    "public_lb": "",
    "submission_file": "",
    "conclusion": (
        f"CV mean {e010_result.cv_mean:.5f} vs E006 {E006_CV_MEAN:.5f} "
        f"(delta {e010_deltas['mean_delta']:+.5f}, {e010_deltas['folds_improved']}/5 folds improved, "
        f"range [{e010_deltas['min_delta']:+.5f}, {e010_deltas['max_delta']:+.5f}]). "
        "None of the 5 folds hit the 2500-iteration ceiling -- genuine early-stopping "
        "convergence, not truncation, confirming E006 was materially constrained by its "
        "800-iteration cap. Strongest and most consistent Build 6 gain so far; becomes "
        "the current tuning control entering Phase 2."
    ),
    "next_action": (
        "Test tree complexity (max_depth, min_child_weight) against this configuration "
        "in Phase 2, screening on a single fold before committing to full CV."
    ),
}
append_experiment_row(e010_row)

append_tuning_row(tuning_result_row(
    "E010", "phase1_full_cv", "E010", E010_PARAMS, e010_result,
    control_scores=E006_FOLDS, runtime_minutes=e010_elapsed / 60,
    decision=e010_row["conclusion"],
))
pd.read_csv(experiments_path).tail(1)

E010 already recorded — skipping append (idempotent re-run).
E010 already recorded — skipping append (idempotent re-run).


,experiment_id,date,model,feature_set,hypothesis,preprocessing,cv_method,seed,parameters,fold_1_auc,fold_2_auc,fold_3_auc,fold_4_auc,fold_5_auc,cv_mean,cv_std,public_lb,submission_file,conclusion,next_action
9,E010,2026-08-21,XGBClassifier,raw + screen_residual,E006 (and E004 before it) frequently hit or ne...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"XGBClassifier(objective=binary:logistic, eval_...",0.96429,0.965,0.96509,0.96585,0.96475,0.96499,0.00051,NaN,NaN,CV mean 0.96499 vs E006 0.96445 (delta +0.0005...,"Test tree complexity (max_depth, min_child_wei..."


## 5. Phase 2 — Tree complexity screening

Single-fold screens of `max_depth` and `min_child_weight` against
E010's fold-1 baseline, holding every other parameter (including
`learning_rate=0.05`, `n_estimators=2500`) fixed. Read from the tuning
tracker (see the Section 3 note on execution).

In [7]:
e010_baseline_auc = e010_result.fold_scores[0]
print(f"E010 fold-1 baseline: {e010_baseline_auc:.5f}")

phase2_screens = show_stage("phase2_screening")
phase2_screens

E010 fold-1 baseline: 0.96429


,trial_or_experiment_id,stage,formal_experiment,learning_rate,n_estimators,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,gamma,cv_mean,cv_std,delta_vs_control,folds_improved,mean_best_iteration,runtime_minutes,decision
0,screen-p2-depth5,phase2_screening,NaN,0.05,2500,5,1,0.9,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,2498.0,8.2,single-fold screen vs E010 fold-1 (0.96429); f...
1,screen-p2-depth7,phase2_screening,NaN,0.05,2500,7,1,0.9,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,1685.0,6.0,single-fold screen vs E010 fold-1 (0.96429); f...
2,screen-p2-mcw3,phase2_screening,NaN,0.05,2500,6,3,0.9,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,2166.0,7.4,single-fold screen vs E010 fold-1 (0.96429); f...


In [8]:
print("Phase 2 conclusion: no gain over E010. max_depth=6, min_child_weight=1 retained unchanged.")

Phase 2 conclusion: no gain over E010. max_depth=6, min_child_weight=1 retained unchanged.


## 6. Phase 3 — Sampling screening

Single-fold screens of `subsample` and `colsample_bytree` against the
same E010 fold-1 baseline. Read from the tuning tracker (see the
Section 3 note on execution).

In [9]:
phase3_screens = show_stage("phase3_screening")
phase3_screens

,trial_or_experiment_id,stage,formal_experiment,learning_rate,n_estimators,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,gamma,cv_mean,cv_std,delta_vs_control,folds_improved,mean_best_iteration,runtime_minutes,decision
0,screen-p3-sub0.8col0.8,phase3_screening,NaN,0.05,2500,6,1,0.8,0.8,0.0,1.0,0.0,NaN,NaN,NaN,NaN,2449.0,8.1,single-fold screen vs E010 fold-1 (0.96429); f...
1,screen-p3-sub1.0col1.0,phase3_screening,NaN,0.05,2500,6,1,1.0,1.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,2207.0,6.1,single-fold screen vs E010 fold-1 (0.96429); f...
2,screen-p3-sub0.7col0.9,phase3_screening,NaN,0.05,2500,6,1,0.7,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,2268.0,5.9,single-fold screen vs E010 fold-1 (0.96429); f...


In [10]:
print("Phase 3 conclusion: no gain over E010. subsample=0.9, colsample_bytree=0.9 retained unchanged.")

Phase 3 conclusion: no gain over E010. subsample=0.9, colsample_bytree=0.9 retained unchanged.


## 7. Phase 4 — Regularization screening

Single-fold screens of `reg_alpha` and `reg_lambda` against the same
E010 fold-1 baseline. Read from the tuning tracker (see the Section 3
note on execution).

In [11]:
phase4_screens = show_stage("phase4_screening")
phase4_screens

,trial_or_experiment_id,stage,formal_experiment,learning_rate,n_estimators,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,gamma,cv_mean,cv_std,delta_vs_control,folds_improved,mean_best_iteration,runtime_minutes,decision
0,screen-p4-alpha0.1,phase4_screening,NaN,0.05,2500,6,1,0.9,0.9,0.1,1.0,0.0,NaN,NaN,NaN,NaN,2427.0,7.6,single-fold screen vs E010 fold-1 (0.96429); f...
1,screen-p4-lambda2,phase4_screening,NaN,0.05,2500,6,1,0.9,0.9,0.0,2.0,0.0,NaN,NaN,NaN,NaN,2268.0,6.4,single-fold screen vs E010 fold-1 (0.96429); f...
2,screen-p4-lambda5,phase4_screening,NaN,0.05,2500,6,1,0.9,0.9,0.0,5.0,0.0,NaN,NaN,NaN,NaN,2485.0,7.9,single-fold screen vs E010 fold-1 (0.96429); f...


In [12]:
print("Phase 4 conclusion: no gain over E010. reg_alpha=0, reg_lambda=1 retained unchanged.")
print("Phases 2/3/4 all found no gain over E010, triggering the Build 6 stopping rule - "
      "joint refinement skipped, E010 adopted as final tuned candidate.")

Phase 4 conclusion: no gain over E010. reg_alpha=0, reg_lambda=1 retained unchanged.
Phases 2/3/4 all found no gain over E010, triggering the Build 6 stopping rule - joint refinement skipped, E010 adopted as final tuned candidate.


## 8. Stopping rule and final candidate decision

**Build 6 stopping rule:** if several consecutive sensible configurations
fail to improve the current best, stop and adopt the current best
directly rather than running a joint-refinement full CV just to confirm
a null result.

**Applied here:** Phase 2 (tree complexity), Phase 3 (sampling), and
Phase 4 (regularization) each tested 3 single-fold candidates against
E010's fold-1 baseline (0.96429) and found nothing exceeding the
~0.0005 single-fold noise band observed throughout this build:

| Phase | Best candidate delta | Verdict |
|---|---|---|
| 2 (tree complexity) | +0.00007 (`min_child_weight=3`) | noise, not promoted |
| 3 (sampling) | -0.00001 (`sub=0.8, col=0.8`) | flat, not promoted |
| 4 (regularization) | +0.00002 (`reg_lambda=5.0`) | noise, not promoted |

Three consecutive phases with no real gain triggers the stopping rule.
**Decision: joint refinement is skipped. E010 (CV mean 0.96499, std
0.00051) is adopted as Build 6's final tuned XGBoost configuration.**

## 9. OOF prediction correlation — E010 vs E006 (diagnostic only)

Diagnostic only — does not trigger any ensembling action (reserved for
Build 7). Compares E010's out-of-fold predictions (Section 4) against
the reconstructed E006 out-of-fold predictions (Section 2).

In [13]:
pearson = float(np.corrcoef(e010_result.oof_predictions, e006_reconstructed.oof_predictions)[0, 1])
spearman = float(pd.Series(e010_result.oof_predictions).corr(
    pd.Series(e006_reconstructed.oof_predictions), method="spearman"
))

print(f"Pearson:  {pearson:.6f}")
print(f"Spearman: {spearman:.6f}")

oof_corr = pd.DataFrame({"E010": [1.0, pearson], "E006": [pearson, 1.0]}, index=["E010", "E006"])
oof_corr.to_csv(OUTPUTS_DIR / "e010_e006_oof_correlation.csv")
oof_corr

Pearson:  0.996747
Spearman: 0.997404


,E010,E006
E010,1.000000,0.996747
E006,0.996747,1.000000


CatBoost/E008's OOF predictions were not saved to disk in Build 4/5 and
were not recomputed here (a full CatBoost 5-fold CV run takes ~40-44
minutes; the CatBoost/E010 correlation is an optional diagnostic, not
required for this build's acceptance criteria). E010 and E006 are, as
expected, extremely highly correlated (Pearson 0.9967) since E010 is the
same architecture and feature set as E006 with only the iteration
budget/learning rate changed — this is a refinement of E006, not a
diverse alternative.

## 10. Final tuned parameters artifact

Writes `outputs/best_xgboost_params.json`, the authoritative record of
Build 6's final tuned XGBoost configuration.

In [14]:
best_params_record = {
    "experiment_id": "E010",
    "model": "XGBClassifier",
    "feature_set": "raw predictors + screen_residual",
    "params": {k: v for k, v in E010_PARAMS.items()},
    "early_stopping_rounds": 50,
    "cv_mean": round(e010_result.cv_mean, 5),
    "cv_std": round(e010_result.cv_std, 5),
    "fold_scores": [round(s, 5) for s in e010_result.fold_scores],
    "best_iterations": e010_result.best_iterations,
    "notes": (
        "Final tuned Build 6 XGBoost configuration. Selected in Phase 1 "
        "(learning_rate/n_estimators sweep); Phases 2 (tree complexity), 3 "
        "(sampling), and 4 (regularization) all found no improvement over this "
        "configuration, so it was adopted directly without joint refinement."
    ),
}

import json
with open(OUTPUTS_DIR / "best_xgboost_params.json", "w") as f:
    json.dump(best_params_record, f, indent=2)

print(json.dumps(best_params_record, indent=2))

{
  "experiment_id": "E010",
  "model": "XGBClassifier",
  "feature_set": "raw predictors + screen_residual",
  "params": {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 1,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "gamma": 0.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "n_estimators": 2500,
    "tree_method": "hist",
    "random_state": 42
  },
  "early_stopping_rounds": 50,
  "cv_mean": 0.96499,
  "cv_std": 0.00051,
  "fold_scores": [
    0.96429,
    0.965,
    0.96509,
    0.96585,
    0.96475
  ],
  "best_iterations": [
    2485,
    2300,
    2445,
    2429,
    2294
  ],
  "notes": "Final tuned Build 6 XGBoost configuration. Selected in Phase 1 (learning_rate/n_estimators sweep); Phases 2 (tree complexity), 3 (sampling), and 4 (regularization) all found no improvement over this configuration, so it was adopted directly without joint refinement."
}


## 11. Submission generation and validation

Reuses `e010_result.test_predictions` (already fold-averaged across the
same 5-fold CV run in Section 4 — no need to refit) to build and
validate the final Kaggle submission.

In [15]:
submission = pd.DataFrame({
    ID_COLUMN: test[ID_COLUMN],
    TARGET_COLUMN: e010_result.test_predictions,
})
validate_submission(submission, sample_submission)

DELIVERABLES_DIR.mkdir(exist_ok=True)
filename = "deliverables/submission_E010_xgb_tuned.csv"
output_path = DELIVERABLES_DIR / "submission_E010_xgb_tuned.csv"
submission.to_csv(output_path, index=False)

df = pd.read_csv(experiments_path)
df["submission_file"] = df["submission_file"].astype(object)
df.loc[df["experiment_id"] == "E010", "submission_file"] = filename
df.to_csv(experiments_path, index=False)

print(f"Wrote {len(submission)} rows to {output_path}, validated OK.")
print("Ready for manual Kaggle upload. Public LB will be recorded once supplied.")

Wrote 296302 rows to D:\Projects\kaggle-smartphone-addiction\deliverables\submission_E010_xgb_tuned.csv, validated OK.
Ready for manual Kaggle upload. Public LB will be recorded once supplied.


## 12. Consolidated tuning results and experiment record

Final state of both trackers after this notebook's run.

In [16]:
print("=== outputs/xgboost_tuning_results.csv (all Build 6 rows) ===")
tuning_df = pd.read_csv(tuning_results_path)
display(tuning_df)

=== outputs/xgboost_tuning_results.csv (all Build 6 rows) ===


,trial_or_experiment_id,stage,formal_experiment,learning_rate,n_estimators,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,gamma,cv_mean,cv_std,delta_vs_control,folds_improved,mean_best_iteration,runtime_minutes,decision
0,screen-p1-baseline,phase1_screening,NaN,0.10,800,6,1,0.9,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,799.0,2.1,single-fold screen only (fold 1); reproduces E...
1,screen-p1-lr0.05,phase1_screening,NaN,0.05,2000,6,1,0.9,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,1994.0,7.2,single-fold screen; fold_auc=0.96417 (+0.00043...
2,screen-p1-lr0.07,phase1_screening,NaN,0.07,1400,6,1,0.9,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,1385.0,4.0,single-fold screen; fold_auc=0.96414 (+0.00040...
3,screen-p1-lr0.03,phase1_screening,NaN,0.03,3200,6,1,0.9,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,3199.0,9.1,single-fold screen; fold_auc=0.96420 (+0.00046...
4,E010,phase1_full_cv,E010,0.05,2500,6,1,0.9,0.9,0.0,1.0,0.0,0.96499,0.00051,0.00055,5.0,2390.6,45.4,"accepted; full 5-fold CV, 5/5 folds improved v..."
5,screen-p2-depth5,phase2_screening,NaN,0.05,2500,5,1,0.9,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,2498.0,8.2,single-fold screen vs E010 fold-1 (0.96429); f...
6,screen-p2-depth7,phase2_screening,NaN,0.05,2500,7,1,0.9,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,1685.0,6.0,single-fold screen vs E010 fold-1 (0.96429); f...
7,screen-p2-mcw3,phase2_screening,NaN,0.05,2500,6,3,0.9,0.9,0.0,1.0,0.0,NaN,NaN,NaN,NaN,2166.0,7.4,single-fold screen vs E010 fold-1 (0.96429); f...
8,screen-p3-sub0.8col0.8,phase3_screening,NaN,0.05,2500,6,1,0.8,0.8,0.0,1.0,0.0,NaN,NaN,NaN,NaN,2449.0,8.1,single-fold screen vs E010 fold-1 (0.96429); f...
9,screen-p3-sub1.0col1.0,phase3_screening,NaN,0.05,2500,6,1,1.0,1.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,2207.0,6.1,single-fold screen vs E010 fold-1 (0.96429); f...


In [17]:
print("=== experiments/experiments.csv (E006, E010) ===")
exp_df = pd.read_csv(experiments_path)
display(exp_df.loc[exp_df["experiment_id"].isin(["E006", "E010"])])

=== experiments/experiments.csv (E006, E010) ===


,experiment_id,date,model,feature_set,hypothesis,preprocessing,cv_method,seed,parameters,fold_1_auc,fold_2_auc,fold_3_auc,fold_4_auc,fold_5_auc,cv_mean,cv_std,public_lb,submission_file,conclusion,next_action
5,E006,2026-08-20,XGBClassifier,raw + screen_residual,The residual may represent screen use not expl...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"XGBClassifier(objective=binary:logistic, eval_...",0.96374,0.96438,0.96453,0.96543,0.96416,0.96445,0.00056,0.96608,deliverables/E006_xgb_screen_residual_submissi...,CV mean 0.96445 vs E004 0.96382 (delta +0.0006...,Transfer-test on CatBoost (E008); include in f...
9,E010,2026-08-21,XGBClassifier,raw + screen_residual,E006 (and E004 before it) frequently hit or ne...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"XGBClassifier(objective=binary:logistic, eval_...",0.96429,0.96500,0.96509,0.96585,0.96475,0.96499,0.00051,NaN,deliverables/submission_E010_xgb_tuned.csv,CV mean 0.96499 vs E006 0.96445 (delta +0.0005...,"Test tree complexity (max_depth, min_child_wei..."


## 13. Build 6 conclusions

- **E006 was materially iteration-constrained.** Reconstructing it
  confirmed `best_iterations` at or one below its 800-iteration cap in
  4/5 folds. Lowering `learning_rate` to 0.05 and raising the ceiling to
  2500 (E010) let early stopping converge genuinely (best_iterations
  2294-2485, no fold hit the ceiling) and improved CV mean by +0.00055
  (0.96445 -> 0.96499), 5/5 folds improved.
- **Tree complexity, sampling, and regularization tuning around E010 all
  found nothing.** Every Phase 2/3/4 single-fold candidate fell within
  the ~0.0005 single-fold noise band; none was promoted to full CV.
- **Per the Build 6 stopping rule, joint refinement was skipped.** Three
  consecutive phases with no real gain is sufficient evidence that
  E010's neighborhood is not improvable with this parameter set — a
  joint-refinement full CV run would only have re-confirmed a null
  result at meaningful compute cost.
- **E010 is Build 6's final tuned XGBoost configuration**
  (`outputs/best_xgboost_params.json`), with a validated submission at
  `deliverables/submission_E010_xgb_tuned.csv` awaiting manual Kaggle
  upload and public LB recording.
- **E010's OOF predictions correlate at Pearson 0.9967 with E006's** —
  expected, since E010 refines E006's exact architecture rather than
  introducing a diverse alternative; this does not motivate any Build 7
  ensembling decision on its own.
- **Deferred, explicitly out of scope for Build 6:** CatBoost iteration-
  budget diagnostic (optional, not run here), LightGBM tuning,
  ensembling/blending/stacking (Build 7), final submission strategy
  (Build 9).